# Do dialects emerge during isolation?

Everyone starts with the same twelve nouns and verb–object order. After a shared period of contact, each seed branches. One population maintains cross-group contact throughout. The other has no cross-group conversations until contact is restored. Speakers can coin words, borrow forms, reverse word order, accommodate a listener, and pass an imperfect sample of their knowledge to successors. The same rules apply everywhere.

The experimental unit is a seed. Positive lexical or grammatical differences between the paired populations mean more regional divergence under the isolation history. A loss of mutual intelligibility appears as a negative difference in first-pass understanding.

In [1]:
import json
from pathlib import Path
from statistics import mean

from simulation import OBJECTS, ORDERS, WORDS, experiment, profile

cfg = json.loads(Path("config.json").read_text())
seeds = cfg["seeds"]
split = cfg["shared_rounds"]
apart = split + cfg["isolation_rounds"]
end = apart + cfg["reconnection_rounds"]

scores = {}
example = {}
for seed in seeds:
    for arm, t, groups, measures in experiment(cfg, seed):
        scores[seed, arm, t] = measures
        if seed == seeds[0] and arm == "isolated":
            example[t] = groups

## What changed across groups?

Lexical distance is the average pairwise total variation distance between groups’ word-use distributions for the same object. Grammatical distance uses the two word orders; here it equals the absolute difference in the groups’ probability of object–verb order, averaged over pairs. Both range from 0 (identical distributions) to 1 (disjoint).

Mutual intelligibility is the proportion of two-token requests understood correctly on the first attempt. Every direction between different groups contributes equally. Speakers use learned forms; the probe has no fresh innovation, audience accommodation, repair, or learning. Probing cannot create contact during isolation. The two action words are shared by design.

In [2]:
print("Round  History      Lexical TV    Order TV   Intelligibility")
for t, arms in [(split, ["shared"]), (apart, ["sustained", "isolated"]), (end, ["sustained", "isolated"])]:
    for arm in arms:
        lex, order, understood = map(mean, zip(*(scores[s, arm, t] for s in seeds)))
        print(f"{t:5}  {arm:10}   {lex:9.4f}   {order:9.4f}   {understood:13.2%}")

Round  History      Lexical TV    Order TV   Intelligibility
   20  shared          0.0043      0.0033          99.45%
  180  sustained       0.0114      0.0080          98.99%
  180  isolated        0.0143      0.0137          98.85%
  260  sustained       0.0093      0.0131          99.18%
  260  isolated        0.0099      0.0129          99.14%


### Paired differences for each seed

Subtract sustained contact from the isolation history at the same round. Recognition differences are in percentage points. The isolation-history population has already reconnected at the second endpoint.

In [3]:
print("             End of isolation                After reconnection")
print("Seed   Lexical    Order  Recogn. pp     Lexical    Order  Recogn. pp")
for seed in seeds:
    parts = []
    for t in (apart, end):
        a, b = scores[seed, "isolated", t], scores[seed, "sustained", t]
        parts.append(f"{a[0]-b[0]:+8.4f} {a[1]-b[1]:+8.4f} {100*(a[2]-b[2]):+10.2f}")
    print(f"{seed:4} " + "    ".join(parts))

             End of isolation                After reconnection
Seed   Lexical    Order  Recogn. pp     Lexical    Order  Recogn. pp
   1  +0.0153  -0.0068      -0.94     +0.0143  -0.0138      -1.44
   2  +0.0070  +0.0145      -0.67     -0.0004  +0.0112      +0.06
   3  +0.0145  +0.0146      -0.50     +0.0046  +0.0027      -0.39
   4  -0.0008  -0.0128      +0.50     -0.0038  -0.0008      -0.28
   5  +0.0037  +0.0002      +0.17     +0.0028  +0.0023      +0.11
   6  -0.0035  +0.0096      -0.06     -0.0059  -0.0104      +0.94
   7  -0.0061  +0.0231      +0.17     -0.0048  +0.0072      +0.56
   8  -0.0075  +0.0031      +0.28     -0.0018  -0.0000      +0.11


## Examples from the first seed

To make small changes visible, pick the two learned non-founding noun forms with the greatest regional difference in production probability at the end of isolation. Follow those exact forms through reconnection, even if they disappear. These are illustrative extremes within the first seed. The preceding tables include every meaning and seed. Percentages are group averages of the speakers’ learned production probabilities.

In [4]:
forms = []
for obj, original in zip(OBJECTS, WORDS):
    groups = [profile(group, obj) for group in example[apart]]
    for word in sorted(set().union(*groups) - {original}):
        shares = [group[word] for group in groups]
        forms.append((max(shares) - min(shares), obj, word))

for spread, obj, word in sorted(forms, reverse=True)[:2]:
    print(f"{obj} (founding form {WORDS[OBJECTS.index(obj)]}, learned variant {word})")
    for t in (apart, end):
        print(f"  round {t}  " + "  ".join(
            f"group {i}: {profile(group, obj)[word]:.1%}" for i, group in enumerate(example[t], 1)))
if not forms:
    print("No non-founding word remained in any group at the end of isolation.")

cup (founding form kupa, learned variant sekumi)
  round 180  group 1: 12.3%  group 2: 0.0%  group 3: 0.0%
  round 260  group 1: 0.9%  group 2: 0.0%  group 3: 0.0%
salt (founding form sala, learned variant sesetu)
  round 180  group 1: 10.5%  group 2: 0.0%  group 3: 0.0%
  round 260  group 1: 13.7%  group 2: 4.4%  group 3: 0.0%


All founders put the verb first (for example, *give taku* for “give stone”). A speaker can reverse that order, and a listener can learn it after a successful interpretation or a pointing repair. The following frequencies show whether that alternative became a regional convention.

In [5]:
for t in (apart, end):
    print(f"Object before verb, round {t}: " + "  ".join(
        f"group {i}: {profile(group)[ORDERS[1]]:.1%}" for i, group in enumerate(example[t], 1)))

Object before verb, round 180: group 1: 0.8%  group 2: 1.0%  group 3: 0.8%
Object before verb, round 260: group 1: 2.7%  group 2: 2.2%  group 3: 2.1%


### Requests these speakers actually produced in a read-only sample

From the first seed at the end of isolation, draw ordinary requests to listeners in another group. Show the first request using a learned non-founding noun and the first using the learned object–verb order. Fresh invention, pointing, and audience accommodation are off in this sample. These are elicited utterances from the saved speakers; the probe leaves their vocabularies alone.

In [6]:
from random import Random
from simulation import ACTIONS, interpret, produce

rng = Random(42)
shown = set()
for _ in range(1000):
    a, b = rng.sample(example[apart], 2)
    action, obj = rng.choice(ACTIONS), rng.choice(OBJECTS)
    tokens, word, order = produce(rng.choice(a), action, obj, rng)
    answer = interpret(rng.choice(b), tokens, rng)
    for label, present in [("borrowed noun", word != WORDS[OBJECTS.index(obj)]),
                           ("object before verb", order == ORDERS[1])]:
        if present and label not in shown:
            print(f"{label}: {' '.join(tokens)!r} meant {action} {obj}; listener interpreted {answer}.")
            shown.add(label)
    if len(shown) == 2:
        break

object before verb: 'pisi give' meant give fish; listener interpreted ('give', 'fish').
borrowed noun: 'show raniku' meant show cup; listener interpreted None.


## Reading the result

The isolation history ended round 180 with slightly greater mean lexical and word-order divergence than sustained contact. Lexical divergence was greater in only four of the eight seed pairs. Average first-pass comprehension stayed above 98% in both histories. By round 260, the mean lexical gap had narrowed and the mean word-order gap was effectively gone.

Some local words took hold, and some remained in use after contact returned. Across the eight seeds there is little evidence here for a consistent split into sharply distinct dialects. The test uses twelve objects, two shared action words, and two possible word orders. It asks whether listeners can recover a simple request under these rules.